In [1]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly import express as px
from plotly.io import write_image

In [ ]:
# csv_dir = 'output_csv/classification_CLS_UWaveGestureLibrary_MambaSingleLayer_UEA_ftM_sl315_ll0_pl0_dm1024_ds2_expand1_dc4_nk7_tvdt0_tvB0_tvC1_useD0_gating4proposed_0/csv/'
# num_classes = 8
# class_list = ['rd-ld', 'square', 'right', 'left', 'up', 'down', 'clockwise', 'counterclockwise']  # order unknown


csv_dir = 'output_csv/classification_CLS_RacketSports_MambaSingleLayer_UEA_ftM_sl30_ll0_pl0_dm1024_ds4_expand1_dc4_nk3_tvdt1_tvB0_tvC1_useD0_gating4proposed_0/csv/'
num_classes = 4
class_list = ['Badminton_Smash', 'Badminton_Clear', 'Squash_ForehandBoast', 'Squash_BackhandBoast']


lst = []
for fname in os.listdir(csv_dir):
  tmp = (fname.split('.')[0]).split('_')
  lst.append([int(tmp[2]), int(tmp[3].replace('label', '')), int(tmp[4].replace('pred', '')), fname])

df_flist = pd.DataFrame(lst, columns=['sample_idx', 'label', 'pred', 'filename']).set_index('sample_idx')
df_flist

,label,pred,filename
sample_idx,,,
150,2,2,CLS_RacketSports_150_label2_pred2.csv
38,1,1,CLS_RacketSports_38_label1_pred1.csv
27,1,0,CLS_RacketSports_27_label1_pred0.csv
135,2,2,CLS_RacketSports_135_label2_pred2.csv
98,3,3,CLS_RacketSports_98_label3_pred3.csv
...,...,...,...
34,1,1,CLS_RacketSports_34_label1_pred1.csv
139,2,2,CLS_RacketSports_139_label2_pred2.csv
94,3,3,CLS_RacketSports_94_label3_pred3.csv


In [3]:
# heatmap

fig = px.density_heatmap(df_flist, x='label', y='pred', nbinsx=num_classes, nbinsy=num_classes, title='Confusion Matrix Heatmap')
fig.update_layout(xaxis_title='True Label', yaxis_title='Predicted Label')

# map axis to category (alphabet a to z)
fig.update_xaxes(tickmode='array', tickvals=list(range(num_classes)), ticktext=class_list)
fig.update_yaxes(tickmode='array', tickvals=list(range(num_classes)), ticktext=class_list)

fig.show()

In [4]:
tmp_fname = df_flist[(df_flist['label'] == 0) & (df_flist['pred'] == 0)].iloc[0]['filename']
output_df = pd.read_csv(os.path.join(csv_dir, tmp_fname))
output_df.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,hidden_0,hidden_1,hidden_2,hidden_3,...,hidden_1019,hidden_1020,hidden_1021,hidden_1022,hidden_1023,logit_0,logit_1,logit_2,logit_3,attn_0
0,-0.167479,-0.141235,0.074315,-0.090260,0.082201,-0.068519,0.024942,0.033206,0.103780,-0.053588,...,-0.036619,0.045565,0.078027,0.034934,0.025778,0.209475,0.620805,-0.453151,-0.284623,0.021471
1,-0.099042,-0.168878,0.013763,0.157887,0.419728,-0.295375,0.067608,0.138222,0.405862,-0.201472,...,-0.091212,0.146211,0.263192,0.153330,0.244705,0.839336,2.085322,-2.023103,-0.825523,0.019957
2,-0.116648,0.006263,-0.780075,-0.020779,0.438959,-0.309855,0.112184,0.274361,0.346925,-0.133603,...,0.006445,0.004049,0.687307,0.226779,0.132136,2.278895,5.884836,-3.999998,-2.959581,0.017277
3,-0.358460,-0.396692,1.297341,-0.023698,0.144534,-0.326404,0.007980,0.489296,0.696238,-0.170725,...,-0.162765,-0.091516,0.469661,0.497083,0.430056,1.486901,8.824745,-4.779601,-4.145691,0.019394
4,-0.544575,0.341953,1.186683,0.070889,0.291083,-0.007841,0.522454,0.738331,0.481404,-0.240978,...,-0.220292,0.091053,0.861144,-0.063192,0.719835,8.229753,6.489767,-6.973183,-5.733115,0.015948


In [5]:

from matplotlib import colormaps
cmap = colormaps['YlGnBu']


for idx in [0,5,20,24,32,39,129]:
    print(f'IDX {idx}')
    tmp_fname = df_flist.loc[idx, 'filename']
    output_df = pd.read_csv(os.path.join(csv_dir, tmp_fname))

    # all different dash lines
    dash_types = ['dot', 'dashdot', 'longdashdot', 'solid']
    line_dash_map = {f'logit_{i}': dash_types[i % len(dash_types)] for i in range(num_classes)}
    fig = px.line(output_df, x=output_df.index, y=[f'logit_{i}' for i in range(num_classes)], 
                title=f'Class label: {class_list[df_flist.loc[idx, "label"]]}',
                labels={'index': 'time step', 'value': 'logit value', 'variable': 'class'},
                # line_dash='variable',
                # line_dash_map=line_dash_map
    )

    # fig size
    fig.update_layout(width=650, height=200)
    # tighten layout
    fig.update_layout(margin=dict(l=20, r=20, t=40, b=20))
    fig.update_xaxes(title_standoff=2)
    fig.update_yaxes(title_standoff=2)

    # update legend values
    fig.for_each_trace(lambda t: t.update(name=class_list[int(t.name.split('_')[1])],
                                          legendgroup=class_list[int(t.name.split('_')[1])],
                                          hovertemplate=t.hovertemplate.replace(t.name, class_list[int(t.name.split('_')[1])])
                                         )
                      )
    # fig.update_layout(legend=dict(
    #     title='Classes',
    #     orientation='h',
    #     yanchor='bottom',
    #     y=1.02,
    #     xanchor='right',
    #     x=1
    # ))
    


    # add attention as background color
    colors = [f'rgba({",".join([f"{i}" for i in cmap(val*5, bytes=True)[:4]])})' for val in output_df[f'attn_0']]
    fig.update_layout(
        shapes=[
            
            go.layout.Shape(
                type="rect",
                # x-reference is assigned to the x-values
                xref="x",
                # y-reference is assigned to the plot paper [0,1]
                yref="paper",
                x0=idx-0.5,
                y0=0,
                x1=idx+0.5,
                y1=1,
                fillcolor=colors[idx],
                opacity=0.5,
                layer="below",
                line_width=0,
            )
            for idx in range(len(colors))
        ]
    )
    fig.show()


    simple_avg_logit = output_df[[f'logit_{i}' for i in range(num_classes)]].mean(axis=0).values
    weighted_avg_logit = (output_df[[f'logit_{i}' for i in range(num_classes)]].T * output_df['attn_0']).T.sum(axis=0).values / output_df['attn_0'].sum()

    print(f'Simple Avg Logit: {simple_avg_logit}, Pred: {class_list[simple_avg_logit.argmax()]}')
    print(f'Weighted Avg Logit: {weighted_avg_logit}, Pred: {class_list[weighted_avg_logit.argmax()]}')
    print(f'is different? {simple_avg_logit.argmax() != weighted_avg_logit.argmax()}\n')
    print(f'a_t: min {output_df["attn_0"].min()}, max {output_df["attn_0"].max()}, mean {output_df["attn_0"].mean()}\n')

    write_image(fig, f'figures_tmp/visualization_gating_idx{idx}_label{df_flist.loc[idx, "label"]}_pred{df_flist.loc[idx, "pred"]}.pdf', 
                width=650, height=200, scale=2)

IDX 0


Simple Avg Logit: [ 5.47335911  2.20355852 -4.18475611 -3.84441874], Pred: Badminton_Smash
Weighted Avg Logit: [ 3.19292618  3.82131955 -3.33364641 -4.30696846], Pred: Badminton_Clear
is different? True

a_t: min 0.015101106, max 0.080967434, mean 0.033333331466666664

IDX 5


Simple Avg Logit: [ 3.08159583  2.8222473  -2.69542473 -3.43149107], Pred: Badminton_Smash
Weighted Avg Logit: [ 0.05235329  4.03271773 -1.47275665 -3.32888127], Pred: Badminton_Clear
is different? True

a_t: min 0.012372452, max 0.11933105, mean 0.033333334533333335

IDX 20


Simple Avg Logit: [ 2.23222436  0.99870969 -1.99829066 -2.29774428], Pred: Badminton_Smash
Weighted Avg Logit: [-0.98418215  2.51870203 -1.00578271 -1.86228853], Pred: Badminton_Clear
is different? True

a_t: min 0.0135663785, max 0.1286542, mean 0.03333333373333334

IDX 24


Simple Avg Logit: [ 5.19986201  1.271118   -3.83539865 -3.0532764 ], Pred: Badminton_Smash
Weighted Avg Logit: [ 2.03803472  2.52265006 -2.65614009 -2.75470647], Pred: Badminton_Clear
is different? True

a_t: min 0.01326449, max 0.12023374, mean 0.03333333345

IDX 32


Simple Avg Logit: [ 3.44459172  2.63803613 -3.81435975 -2.62325369], Pred: Badminton_Smash
Weighted Avg Logit: [-0.78257566  5.34675822 -2.38105554 -2.71969013], Pred: Badminton_Clear
is different? True

a_t: min 0.013206761, max 0.154425, mean 0.033333333

IDX 39


Simple Avg Logit: [ 1.54194722  5.42550508 -4.09045631 -3.09093815], Pred: Badminton_Clear
Weighted Avg Logit: [-2.78146809  7.3759076  -2.40144958 -2.7651155 ], Pred: Badminton_Clear
is different? False

a_t: min 0.011967095, max 0.14674902, mean 0.033333334433333334

IDX 129


Simple Avg Logit: [ 0.03614954  0.91779673  0.54523032 -2.80624769], Pred: Badminton_Clear
Weighted Avg Logit: [-2.0802786   0.59906558  2.64643602 -2.65352882], Pred: Squash_ForehandBoast
is different? True

a_t: min 0.015121391, max 0.08519241, mean 0.03333333386666667



In [10]:

from matplotlib import colormaps
cmap = colormaps['YlGnBu']


# for idx in range(len(df_flist)):
#     if idx in [0,5,20,24,32,39,129]:
#         continue

#     if df_flist.loc[idx, 'label'] != df_flist.loc[idx, 'pred']:
#         continue

for idx in [15, 17, 41, 64, 70, 124, 135, 140, 148]:  # max less than 0.05

    print(f'IDX {idx}')
    tmp_fname = df_flist.loc[idx, 'filename']
    output_df = pd.read_csv(os.path.join(csv_dir, tmp_fname))

    # all different dash lines
    dash_types = ['dot', 'dashdot', 'longdashdot', 'solid']
    line_dash_map = {f'logit_{i}': dash_types[i % len(dash_types)] for i in range(num_classes)}
    fig = px.line(output_df, x=output_df.index, y=[f'logit_{i}' for i in range(num_classes)], 
                title=f'Class label: {class_list[df_flist.loc[idx, "label"]]}',
                labels={'index': 'time step', 'value': 'logit value', 'variable': 'class'},
                # line_dash='variable',
                # line_dash_map=line_dash_map
    )

    # fig size
    fig.update_layout(width=650, height=200)
    # tighten layout
    fig.update_layout(margin=dict(l=20, r=20, t=40, b=20))
    fig.update_xaxes(title_standoff=2)
    fig.update_yaxes(title_standoff=2)

    # update legend values
    fig.for_each_trace(lambda t: t.update(name=class_list[int(t.name.split('_')[1])],
                                          legendgroup=class_list[int(t.name.split('_')[1])],
                                          hovertemplate=t.hovertemplate.replace(t.name, class_list[int(t.name.split('_')[1])])
                                         )
                      )
    # fig.update_layout(legend=dict(
    #     title='Classes',
    #     orientation='h',
    #     yanchor='bottom',
    #     y=1.02,
    #     xanchor='right',
    #     x=1
    # ))
    


    # add attention as background color
    colors = [f'rgba({",".join([f"{i}" for i in cmap(val*5, bytes=True)[:4]])})' for val in output_df[f'attn_0']]
    fig.update_layout(
        shapes=[
            
            go.layout.Shape(
                type="rect",
                # x-reference is assigned to the x-values
                xref="x",
                # y-reference is assigned to the plot paper [0,1]
                yref="paper",
                x0=idx-0.5,
                y0=0,
                x1=idx+0.5,
                y1=1,
                fillcolor=colors[idx],
                opacity=0.5,
                layer="below",
                line_width=0,
            )
            for idx in range(len(colors))
        ]
    )
    fig.show()


    simple_avg_logit = output_df[[f'logit_{i}' for i in range(num_classes)]].mean(axis=0).values
    weighted_avg_logit = (output_df[[f'logit_{i}' for i in range(num_classes)]].T * output_df['attn_0']).T.sum(axis=0).values / output_df['attn_0'].sum()

    print(f'Simple Avg Logit: {simple_avg_logit}, Pred: {class_list[simple_avg_logit.argmax()]}')
    print(f'Weighted Avg Logit: {weighted_avg_logit}, Pred: {class_list[weighted_avg_logit.argmax()]}')
    print(f'is different? {simple_avg_logit.argmax() != weighted_avg_logit.argmax()}\n')
    print(f'a_t: min {output_df["attn_0"].min()}, max {output_df["attn_0"].max()}, mean {output_df["attn_0"].mean()}\n')

    write_image(fig, f'figures_tmp/visualization_gating_idx{idx}_label{df_flist.loc[idx, "label"]}_pred{df_flist.loc[idx, "pred"]}.pdf', 
                width=650, height=200, scale=2)

IDX 15


Simple Avg Logit: [ 1.33272196  4.34374114 -3.38736475 -3.02162074], Pred: Badminton_Clear
Weighted Avg Logit: [ 1.44029269  4.29673307 -3.44525741 -3.1678533 ], Pred: Badminton_Clear
is different? False

a_t: min 0.020499626, max 0.044947427, mean 0.033333333533333336

IDX 17


Simple Avg Logit: [ 0.16451182  7.33092784 -3.22350227 -4.66433412], Pred: Badminton_Clear
Weighted Avg Logit: [-0.06127435  7.45108324 -3.1761857  -4.66977182], Pred: Badminton_Clear
is different? False

a_t: min 0.020782229, max 0.04476329, mean 0.03333333346666666

IDX 41


Simple Avg Logit: [ 5.49667358 -3.74812037 -3.43253315  0.71629097], Pred: Badminton_Smash
Weighted Avg Logit: [ 4.73990655 -3.6260829  -3.22813193  1.08133865], Pred: Badminton_Smash
is different? False

a_t: min 0.023343695, max 0.04789822, mean 0.033333335433333326

IDX 64


Simple Avg Logit: [ 9.47118742 -3.89157432 -4.82017428 -1.38808772], Pred: Badminton_Smash
Weighted Avg Logit: [10.02050878 -4.95343364 -4.72859547 -1.18602188], Pred: Badminton_Smash
is different? False

a_t: min 0.022809632, max 0.04770652, mean 0.033333334099999994

IDX 70


Simple Avg Logit: [ 6.38952427  0.88653247 -4.15838997 -3.43159488], Pred: Badminton_Smash
Weighted Avg Logit: [ 6.35718778  0.10939914 -3.8420818  -3.10917394], Pred: Badminton_Smash
is different? False

a_t: min 0.018713398, max 0.049299683, mean 0.03333333413333333

IDX 124


Simple Avg Logit: [-6.96194363 -1.53172604  8.9189117  -2.04435976], Pred: Squash_ForehandBoast
Weighted Avg Logit: [-7.04622814 -1.96407034  9.58033068 -2.15412579], Pred: Squash_ForehandBoast
is different? False

a_t: min 0.010318441, max 0.04809623, mean 0.033333331566666666

IDX 135


Simple Avg Logit: [-6.09811491 -2.36169627  8.59556209 -1.59324819], Pred: Squash_ForehandBoast
Weighted Avg Logit: [-6.80511502 -2.3915421   9.63869484 -1.85884867], Pred: Squash_ForehandBoast
is different? False

a_t: min 0.011631576, max 0.0494567, mean 0.03333333536666667

IDX 140


Simple Avg Logit: [-5.17995628 -1.14929635  8.55032574 -3.44040636], Pred: Squash_ForehandBoast
Weighted Avg Logit: [-6.26003757 -1.13029803  9.74618054 -3.55797149], Pred: Squash_ForehandBoast
is different? False

a_t: min 0.01111018, max 0.048171017, mean 0.033333334433333334

IDX 148


Simple Avg Logit: [-4.55110311 -1.92107275  7.57674168 -2.1941658 ], Pred: Squash_ForehandBoast
Weighted Avg Logit: [-6.19452092 -1.59511793  9.06988304 -2.41535718], Pred: Squash_ForehandBoast
is different? False

a_t: min 0.0121504245, max 0.048473526, mean 0.033333336416666665



In [7]:

from matplotlib import colormaps
cmap = colormaps['YlGnBu']

cmap(0.0)
cmap(1.0)

# tmp plot to get colormap

fig = go.Figure(data=go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='markers',
    marker=dict(
        size=20,
        color=[0.05, 0.772125], # set color to an array/list of desired values
        colorscale='YlGnBu', # choose a colorscale
        showscale=True,
        colorbar=dict(
            title='a_t',
            x=1, # Position of the colorbar
        ),
    )
))


# fig size
fig.update_layout(width=650, height=200)
# tighten layout
fig.update_layout(margin=dict(l=20, r=20, t=40, b=20))
fig.update_xaxes(title_standoff=2)
fig.update_yaxes(title_standoff=2)

# add colorbar
fig.update_layout(
    coloraxis=dict(
        colorscale='YlGn',
        cmin=0,
        cmax=100,
        colorbar=dict(
            title='Shared Color Scale',
            x=0.45, # Position of the colorbar
        )
    )
)

fig.show()

write_image(fig, f'figures_tmp/colormap_YlGnBu.pdf', width=650, height=200, scale=2)